In [1]:
# Compact stationarity test function for time series data.

import warnings
import numpy as np

from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tools.sm_exceptions import InterpolationWarning


def stationarity_test(x):

    x = x.dropna()

    if x.nunique() <= 2:

        return {
            "adf_p": np.nan,
            "kpss_p": np.nan,
            "stationary": False
        }

    adf_p = adfuller(
        x,
        autolag="AIC"   # AIC is better but need to confirm the reason
    )[1]

    with warnings.catch_warnings():

        warnings.simplefilter(
            "ignore",
            InterpolationWarning
        )

        try:
            kpss_p = kpss(
                x,
                regression="c",
                nlags="auto"
            )[1]

        except:
            kpss_p = np.nan

    return {
        "adf_p": adf_p,
        "kpss_p": kpss_p,
        "stationary":
            (adf_p < 0.05)
            and
            (kpss_p > 0.05)
    }

In [2]:
# version 1: removing weekend data and combining Friday + Saturday + Sunday into Friday
def create_change_series(folder: Path):

    fixed_file = (
        folder /
        "gt_stitched_fixed_2022-01-01_2026-05-31.csv"
    )

    df = pd.read_csv(
        fixed_file,
        parse_dates=["Time"]
    )


    value_col = [
        c for c in df.columns
        if c != "Time"
    ][0]


    ts = (
        df
        .set_index("Time")[value_col]
        .sort_index()
        .astype(float)
    )


    # -----------------------------
    # Diagnostics
    # -----------------------------

    zero_share = (ts <= 0).mean()

    flat_share = (
        ts.diff()
        .fillna(0)
        .eq(0)
        .mean()
    )

    # -----------------------------
    # Create both transformations
    # -----------------------------

    diff = (
        ts.diff()
        .dropna()
    )

    diff.name = value_col


    log_diff = (
        np.log1p(ts)
        .diff()
        .dropna()
    )

    log_diff.name = value_col


    # -----------------------------
    # Weekend adjustment
    # Combine Friday + Saturday + Sunday
    # because S&P500 is closed on weekends
    # -----------------------------

    def friday_weekend_adjust(series):

        series = series.copy()

        for date in series.index:

            if date.dayofweek == 4:   # Friday

                weekend_dates = [
                    date,
                    date + pd.Timedelta(days=1),
                    date + pd.Timedelta(days=2)
                ]

                values = series.reindex(
                    weekend_dates
                )

                # cumulative weekend information
                series.loc[date] = (
                    values.mean()
                )


        # remove Saturday/Sunday
        series = series[
            ~series.index.dayofweek.isin([5, 6])
        ]


        return series


    diff = friday_weekend_adjust(
        diff
    )

    log_diff = friday_weekend_adjust(
        log_diff
    )



    # -----------------------------
    # Stationarity tests
    # -----------------------------

    level_test = stationarity_test(ts)

    diff_test = stationarity_test(
        diff
    )

    log_diff_test = stationarity_test(
        log_diff
    )



    # -----------------------------
    # Save both outputs
    # -----------------------------

    diff.reset_index().to_csv(
        folder /
        "gt_diff_friday.csv",
        index=False
    )


    log_diff.reset_index().to_csv(
        folder /
        "gt_log_diff_friday.csv",
        index=False
    )

    # -----------------------------
    # Diagnostic text
    # -----------------------------

    lines = [

        "Google Trends Transformation Diagnostic",
        "=" * 60,

        f"Term: {folder.name}",
        "",

        "Original series:",
        f"Observations       : {len(ts)}",
        f"Date range         : {ts.index.min().date()} → {ts.index.max().date()}",
        f"Missing values     : {ts.isna().sum()}",
        "",

        "Zero / flat behavior:",
        f"Zero share         : {zero_share:.4f}",
        f"Flat share         : {flat_share:.4f}",
        "",

        "Level stationarity:",
        str(level_test),
        "",

        "First difference stationarity:",
        str(diff_test),
        "",

        "Log difference stationarity:",
        str(log_diff_test),

    ]


    (folder /
     "transform_diagnostic_friday.txt"
    ).write_text(
        "\n".join(lines),
        encoding="utf-8"
    )


    return {

        "term": folder.name,

        "observations": len(ts),

        "zero_share": round(
            zero_share,
            4
        ),

        "flat_share": round(
            flat_share,
            4
        ),

        "level_stationary":
            level_test["stationary"],

        "diff_stationary":
            diff_test["stationary"],

        "log_diff_stationary":
            log_diff_test["stationary"]

    }

In [3]:
# Version 2: keeping weekends
def create_change_series(folder: Path):

    fixed_file = (
        folder /
        "gt_stitched_fixed_2022-01-01_2026-05-31.csv"
    )

    df = pd.read_csv(
        fixed_file,
        parse_dates=["Time"]
    )


    value_col = [
        c for c in df.columns
        if c != "Time"
    ][0]


    ts = (
        df
        .set_index("Time")[value_col]
        .sort_index()
        .astype(float)
    )


    # -----------------------------
    # Diagnostics
    # -----------------------------

    zero_share = (ts <= 0).mean()

    flat_share = (
        ts.diff()
        .fillna(0)
        .eq(0)
        .mean()
    )

    # -----------------------------
    # Create both transformations
    # -----------------------------

    diff = (
        ts.diff()
        .dropna()
    )

    diff.name = value_col


    log_diff = (
        np.log1p(ts)
        .diff()
        .dropna()
    )

    log_diff.name = value_col


    # -----------------------------
    # Stationarity tests
    # -----------------------------

    level_test = stationarity_test(ts)

    diff_test = stationarity_test(
        diff
    )

    log_diff_test = stationarity_test(
        log_diff
    )



    # -----------------------------
    # Save both outputs
    # -----------------------------

    diff.reset_index().to_csv(
        folder /
        "gt_diff.csv",
        index=False
    )


    log_diff.reset_index().to_csv(
        folder /
        "gt_log_diff.csv",
        index=False
    )

    # -----------------------------
    # Diagnostic text
    # -----------------------------

    lines = [

        "Google Trends Transformation Diagnostic",
        "=" * 60,

        f"Term: {folder.name}",
        "",

        "Original series:",
        f"Observations       : {len(ts)}",
        f"Date range         : {ts.index.min().date()} → {ts.index.max().date()}",
        f"Missing values     : {ts.isna().sum()}",
        "",

        "Zero / flat behavior:",
        f"Zero share         : {zero_share:.4f}",
        f"Flat share         : {flat_share:.4f}",
        "",

        "Level stationarity:",
        str(level_test),
        "",

        "First difference stationarity:",
        str(diff_test),
        "",

        "Log difference stationarity:",
        str(log_diff_test),

    ]


    (folder /
     "transform_diagnostic.txt"
    ).write_text(
        "\n".join(lines),
        encoding="utf-8"
    )


    return {

        "term": folder.name,

        "observations": len(ts),

        "zero_share": round(
            zero_share,
            4
        ),

        "flat_share": round(
            flat_share,
            4
        ),

        "level_stationary":
            level_test["stationary"],

        "diff_stationary":
            diff_test["stationary"],

        "log_diff_stationary":
            log_diff_test["stationary"]

    }

In [ ]:
from pathlib import Path
import pandas as pd


DATA_DIR = Path(
    r"C:\Python\CSUREMM\data_primitive"
)


stationarity_summary = []
failed = []


for folder in DATA_DIR.iterdir():

    if not folder.is_dir():
        continue

    try:

        result = create_change_series(
            folder
        )

        stationarity_summary.append(
            result
        )

        print(
            "done:",
            folder.name
        )


    except Exception as e:

        failed.append(
            (
                folder.name,
                str(e)
            )
        )

        print(
            "failed:",
            folder.name,
            e
        )


# -----------------------------
# Save batch summaries
# -----------------------------

summary = pd.DataFrame(
    stationarity_summary
)


summary.to_csv(
    DATA_DIR /
    "gt_stationarity_summary.csv",
    index=False
)


with open(
    DATA_DIR /
    "gt_stationarity_failures.txt",
    "w",
    encoding="utf-8"
) as f:

    for term, error in failed:

        f.write(
            f"{term}: {error}\n"
        )


print()
print(
    f"Completed: {len(summary)} terms"
)

print(
    f"Failed: {len(failed)} terms"
)

done: accrue
done: advantage
done: affluence
done: affluent
done: afloat
done: allowance
done: aristocracy
done: aristocrat
done: aristocratic
done: associate
done: backer
done: backward


c:\Python\CSUREMM\.venv\Lib\site-packages\statsmodels\tsa\stattools.py:2182: RuntimeWarning: divide by zero encountered in scalar divide
  s_hat = s1 / s0


done: backwardness
done: bankrupt
done: bankruptcy
done: bargain
done: beggar
done: benefactor
done: beneficiary
done: benefit
done: benevolence
done: benevolent
done: bequeath
done: betroth


c:\Python\CSUREMM\.venv\Lib\site-packages\statsmodels\tsa\stattools.py:2182: RuntimeWarning: divide by zero encountered in scalar divide
  s_hat = s1 / s0
c:\Python\CSUREMM\.venv\Lib\site-packages\statsmodels\tsa\stattools.py:2182: RuntimeWarning: divide by zero encountered in scalar divide
  s_hat = s1 / s0


done: betrothal
done: blackmail
done: bonus
done: boom
done: breadwinner
done: bribe
done: broke
done: bum
done: buy
done: capitalize
done: charitable
done: charity
done: cheap
done: colony
done: commoner
done: community
done: compensate
done: compensation
done: contribute
done: contribution
done: cooperative
done: corrupt
done: cost
done: costly
done: crisis
done: debtor
done: default
done: deficit
done: depreciation
done: depression
done: destitute
done: domination
done: donate
done: donation
done: economize
done: endow
done: entrepreneurial
done: equity
done: expense
done: expensive
done: extravagant
done: fellowship
done: fine
done: fire
done: frugal
done: gain
done: gamble
done: generosity
done: ghetto
done: gift
done: gold
done: guide
done: hole
done: hustle
done: hustler
done: inexpensive
done: inflation
done: inherit
done: intervention
done: invaluable
done: jobless
done: laid
done: lay
done: legal
done: liquidate
done: liquidation
done: lucrative
done: luxury
done: meritorious

In [5]:
# batch all words

def build_gt_panel(
    data_dir: Path,
    file_name: str,
    output_name: str
):

    series_list = []


    for folder in sorted(data_dir.iterdir()):

        if not folder.is_dir():
            continue


        file = folder / file_name


        if not file.exists():
            continue


        try:

            df = pd.read_csv(
                file,
                parse_dates=["Time"]
            )


            value_col = [
                c for c in df.columns
                if c != "Time"
            ][0]


            s = (
                df
                .set_index("Time")[value_col]
                .rename(folder.name)
            )


            series_list.append(s)


        except Exception as e:

            print(
                f"failed: {folder.name} | {e}"
            )


    if len(series_list) == 0:
        raise ValueError(
            f"No {file_name} files found"
        )


    panel = pd.concat(
        series_list,
        axis=1,
        join="outer"
    )


    panel = (
        panel
        .sort_index()
    )


    # remove duplicate dates if any
    panel = (
        panel
        .loc[
            ~panel.index.duplicated()
        ]
    )


    output = data_dir / output_name


    panel.to_csv(
        output,
        index=True
    )


    print(
        f"Saved {output_name}"
    )

    print(
        f"Terms: {panel.shape[1]}"
    )

    print(
        f"Days: {panel.shape[0]}"
    )

    print(
        f"Missing cells: {panel.isna().sum().sum()}"
    )


    return panel



# -----------------------------
# Build two GT master panels
# -----------------------------


gt_diff_panel = build_gt_panel(
    DATA_DIR,
    "gt_diff.csv",
    "gt_diff_panel.csv"
)


gt_logdiff_panel = build_gt_panel(
    DATA_DIR,
    "gt_log_diff.csv",
    "gt_log_diff_panel.csv"
)

Saved gt_diff_panel.csv
Terms: 150
Days: 1611
Missing cells: 0
Saved gt_log_diff_panel.csv
Terms: 150
Days: 1611
Missing cells: 0


In [6]:
# ------------------------------------
# Save master stationarity diagnosis
# ------------------------------------

stationarity_panel = pd.DataFrame(
    stationarity_summary
)


stationarity_panel = (
    stationarity_panel
    .sort_values(
        "term"
    )
    .reset_index(
        drop=True
    )
)


output_file = (
    DATA_DIR /
    "stationarity_summary.csv"
)


stationarity_panel.to_csv(
    output_file,
    index=False
)


print(
    f"Saved: {output_file}"
)


print(
    stationarity_panel.head()
)

Saved: C:\Python\CSUREMM\test\stationarity_summary.csv
        term  observations  zero_share  flat_share  level_stationary  \
0     accrue          1612      0.0031      0.0279             False   
1  advantage          1612      0.0000      0.0651             False   
2  affluence          1612      0.4119      0.2835             False   
3   affluent          1612      0.0000      0.0440             False   
4     afloat          1612      0.1253      0.0471              True   

   diff_stationary  log_diff_stationary  
0             True                 True  
1             True                 True  
2             True                 True  
3             True                 True  
4             True                 True  
